In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
hidden_layers=[1024,1024]
epochs=1000
act_func=tf.nn.relu
input_dropout=0.2
hidden_dropout=0.5
learning_rate=0.0001
norm='norm'

In [4]:
train_features, val_features, _, _, train_targets, val_targets, _, _ = load(norm=norm)

print("Training features shape:", train_features.shape)
print("Validation features shape:", val_features.shape)
print("Training targets shape:", train_targets.shape)
print("Validation targets shape:", val_targets.shape)

print("NaN in train_features:", np.isnan(train_features).any())
print("Inf in train_features:", np.isinf(train_features).any())
print("NaN in train_targets:", np.isnan(train_targets).any())
print("Inf in train_targets:", np.isinf(train_targets).any())

Training features shape: (13884, 7060)
Validation features shape: (4614, 7060)
Training targets shape: (13884, 1)
Validation targets shape: (4614, 1)
NaN in train_features: False
Inf in train_features: False
NaN in train_targets: False
Inf in train_targets: False


In [5]:
model = Sequential()
for i, units in enumerate(hidden_layers):
    if i == 0:
        model.add(Dense(
            units,
            input_shape=(train_features.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'))
        if input_dropout > 0:
            model.add(Dropout(float(input_dropout)))
    else:
        model.add(Dense(
            units,
            activation=act_func,
            kernel_initializer='he_normal'))
        if hidden_dropout > 0:
            model.add(Dropout(float(hidden_dropout)))
model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))

I0000 00:00:1771515869.710679  142055 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771515869.800960  142055 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771515869.800991  142055 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771515869.803276  142055 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771515869.803294  142055 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [6]:
optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.5)
model.compile(loss='mean_squared_error', optimizer=optimizer)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1024)              7230464   
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 1024)              1049600   
                                                                 
 dropout_1 (Dropout)         (None, 1024)              0         
                                                                 
 dense_2 (Dense)             (None, 1)                 1025      
                                                                 
Total params: 8281089 (31.59 MB)
Trainable params: 8281089 (31.59 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = Path("checkpoints/final_model.h5")

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=150,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )
]

In [8]:
history = model.fit(
    train_features, train_targets,
    validation_data=(val_features, val_targets),
    epochs=epochs,
    batch_size=64,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

Epoch 1/1000


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

 70/217 [========>.....................] - ETA: 0s - loss: 498.1270

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


206/217 [===========================>..] - ETA: 0s - loss: 451.8506
Epoch 1: val_loss improved from inf to 376.53003, saving model to checkpoints/final_model.h5


/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


217/217 [==============================] - 2s 5ms/step - loss: 452.9944 - val_loss: 376.5300
Epoch 2/1000
217/217 [==============================] - ETA: 0s - loss: 397.6012
Epoch 2: val_loss improved from 376.53003 to 350.61469, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 397.6012 - val_loss: 350.6147
Epoch 3/1000
211/217 [============================>.] - ETA: 0s - loss: 367.8163
Epoch 3: val_loss improved from 350.61469 to 347.87155, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 370.0744 - val_loss: 347.8716
Epoch 4/1000
216/217 [============================>.] - ETA: 0s - loss: 348.6165
Epoch 4: val_loss improved from 347.87155 to 340.61502, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 349.3484 - val_loss: 340.6150
Epoch 5/1000
203/217 [===========================>..] - ETA: 0s - loss: 335.4089
Epoch 5

In [9]:
print("Final training loss:", history.history['loss'][-1])
print("Final validation loss:", history.history['val_loss'][-1])
print(f"\nBest Training Loss: {min(history.history['loss'])}")
print(f"\nBest Validation Loss: {min(history.history['val_loss'])}")

Final training loss: 135.46221923828125
Final validation loss: 289.1243896484375

Best Training Loss: 133.43389892578125

Best Validation Loss: 286.676513671875


In [10]:
if checkpoint_path.exists():
    print("Loading best model from checkpoint...")
    model = tf.keras.models.load_model(str(checkpoint_path))
else:
    print("Checkpoint not found. Using current model.")

val_predictions = model.predict(val_features)
val_targets_flat = val_targets.flatten()
val_predictions_flat = val_predictions.flatten()

mae = mean_absolute_error(val_targets_flat, val_predictions_flat)
mse = mean_squared_error(val_targets_flat, val_predictions_flat)
rmse = np.sqrt(mse)
pearson_corr, _ = pearsonr(val_targets_flat, val_predictions_flat)

print("Validation Metrics:")
print(f"Mean absolute error               : {mae:.4f}")
print(f"Mean squared error                : {mse:.4f}")
print(f"Root mean squared error           : {rmse:.4f}")
print(f"Pearson's correlation coefficient : {pearson_corr:.4f}")

Loading best model from checkpoint...
145/145 [==============================] - 0s 961us/step
Validation Metrics:
Mean absolute error               : 11.3033
Mean squared error                : 286.6765
Root mean squared error           : 16.9315
Pearson's correlation coefficient : 0.6210


In [ ]:
#fold 0 (test fold metrics) 
_, _, _, test_features, _, _, _, test_targets = load(norm=norm)
test_predictions = model.predict(test_features).flatten()
test_targets_flat = test_targets.flatten()
test_pearson, _ = pearsonr(test_targets_flat, test_predictions)
test_mae = mean_absolute_error(test_targets_flat, test_predictions)
print("Test Fold (Fold 0) Metrics:")
print(f"Pearson's correlation coefficient : {test_pearson:.4f}")
print(f"Mean absolute error               : {test_mae:.4f}")

143/143 [==============================] - 0s 1ms/step
Test Fold (Fold 0) Metrics:
Pearson's correlation coefficient : 0.5878
Mean absolute error               : 11.0352
